In [1]:
import pandas as pd

## Set Directories 
data_dir = './data_files'
output_dir = './data_files'

In [2]:
## Import manually extracted data for comparison
manual_extracted_data_df = pd.read_csv(f"{data_dir}/manual_wagner_extractions.csv")
manual_extracted_data_df['species_id'] = (
    manual_extracted_data_df['family'].astype(str) + '_' +
    manual_extracted_data_df['genus'].astype(str) + '_' +
    manual_extracted_data_df['species'].astype(str)
)

manual_extracted_data_df.drop('Unnamed: 0.1', axis=1, inplace=True)
manual_extracted_data_df.drop('Unnamed: 0', axis=1, inplace=True)

manual_extracted_data_df.head()

,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,leaf_hair_description,...,male_corrola_lobes_length,male_corrola_lobes_width,fruit_length,fruit_width,fruit_diameter,seeds_perfruit,seed_length,seed_width,seed_diameter,species_id
0,Asteraceae,Ambrosia,artemisiifolia,common ragweed,pg 256-257,['DICOTS'],NaN,['HIRSUTE'],"['OPPOSITE', 'ALTERNATE']",NaN,...,NaN,NaN,NaN,NaN,NaN,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...",NaN,NaN,NaN,Asteraceae_Ambrosia_artemisiifolia
1,Asteraceae,Dubautia,laxa,na`ena`e pua melemele,"pg 292-295,301",['DICOTS'],pseudoplantaginea,"['GLABROUS (SMOOTH)', 'HISPID']",['WHORLED'],glabrous,...,NaN,NaN,"{'exmin': nan, 'min': 2.0, 'max': 3.0, 'exmax'...",NaN,NaN,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...",NaN,NaN,NaN,Asteraceae_Dubautia_laxa
2,Asteraceae,Tetramolopium,filiforme,no common name,"pg 361-362, 365, 366",['DICOTS'],polyphyllum,['GLABROUS (SMOOTH)'],['ALTERNATE'],NaN,...,NaN,NaN,"{'exmin': nan, 'min': 2.0, 'max': 2.7, 'exmax'...","{'exmin': nan, 'min': 0.6, 'max': nan, 'exmax'...",NaN,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...",NaN,NaN,NaN,Asteraceae_Tetramolopium_filiforme
3,Asteraceae,Encelia,farinosa,brittle bush,pg 312-313,['DICOTS'],NaN,['PUBERULENT'],['ALTERNATE'],densely white tomentose,...,NaN,NaN,"{'exmin': nan, 'min': 4.0, 'max': 4.5, 'exmax'...",NaN,NaN,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...",NaN,NaN,NaN,Asteraceae_Encelia_farinosa
4,Aristolochiaceae,Aristolochia,littoralis,calico flower,"pg 237-238,239",['DICOTS'],NaN,['NAN'],['ALTERNATE'],NaN,...,NaN,NaN,"{'exmin': nan, 'min': nan, 'max': 45.0, 'exmax...",NaN,"{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': 0.6, 'unit':...",NaN,NaN,Aristolochiaceae_Aristolochia_littoralis


In [3]:
man_df_test = manual_extracted_data_df[(manual_extracted_data_df['family'] == 'Aizoaceae') & (manual_extracted_data_df['genus'] == 'Trianthema')]
man_df_test

,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,leaf_hair_description,...,male_corrola_lobes_length,male_corrola_lobes_width,fruit_length,fruit_width,fruit_diameter,seeds_perfruit,seed_length,seed_width,seed_diameter,species_id
51,Aizoaceae,Trianthema,portulacastrum,no common name,pg 178-179,['DICOTS'],NaN,"['GLABROUS (SMOOTH)', 'PUBERULENT']",['OPPOSITE'],glabrous to sparsely pubescent,...,NaN,NaN,NaN,NaN,NaN,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...",NaN,"{'min': 0.15, 'max': 0.2, 'unit': 'cm'}",NaN,Aizoaceae_Trianthema_portulacastrum


In [4]:
import re, ast, pandas as pd, json

with open("/Users/williamharrigan/Desktop/test.txt", "r", encoding="utf-8") as f:
    raw = f.read().strip()

m = re.match(r'^\s*\w+\s*=\s*(\{.*)\s*$', raw, flags=re.DOTALL)
expr = m.group(1) if m else raw

expr = re.sub(r"<[^:>]+:\s*'([^']+)'>", r"'\1'", expr)

class CallsToDicts(ast.NodeTransformer):
    def visit_Call(self, node: ast.Call):
        self.generic_visit(node)
        keys, vals = [], []
        for kw in node.keywords:
            if kw.arg is None: 
                continue
            keys.append(ast.Constant(kw.arg))
            vals.append(kw.value)
        return ast.Dict(keys=keys, values=vals)

tree = ast.parse(expr, mode="eval")
tree = CallsToDicts().visit(tree)
ast.fix_missing_locations(tree)
data = eval(compile(tree, "<cleaned>", "eval"), {}, {})

json_data = json.loads(json.dumps(data, default=lambda o: getattr(o, 'value', str(o))))
df = pd.json_normalize(json_data)

# 🔹 Clean up column names
df.columns = [re.sub(r'^\w+.?', '', c) for c in df.columns]
df.columns = [re.sub(r'^\w+_measurements\.?', '', c) for c in df.columns]

print(df.head())


      family       genus         species infraspecific_epithet common_name  \
0  Aizoaceae  Trianthema  portulacastrum                  None        None   

  hawaiian_name wagner_pg_number description    life_form_type  \
0          None             None      Dicots  [PERENNIAL_HERB]   

  stem_height.min  ...  leaflets_leaf_shape_type leaflets_leaf_margin_type  \
0            None  ...                      None                      None   

  leaflets_leaf_length leaflets_leaf_width island_type         origin  \
0                 None                None      [OAHU]  [NATURALIZED]   

          status ploidy chromosome_number average_chromosome_number  
0  [NATURALIZED]   [2n]          [26, 28]                      27.0  

[1 rows x 243 columns]


In [14]:
import re
import numpy as np
import pandas as pd

# Map suffixes to desired dict keys
suffix_map = {
    'extreme_min': 'exmin',
    'min': 'min',
    'max': 'max',
    'extreme_max': 'exmax'
}

# Regex to capture base and suffix (e.g., staminate_pedicel_length.min)
pat = re.compile(r'^(?P<base>.+?)\.(?P<suffix>extreme_min|min|max|extreme_max)$')

# Find all collapsible column groups
groups = {}  # base -> list of (colname, suffix)
for col in df.columns:
    m = pat.match(col)
    if m:
        base = m.group('base')
        suf = m.group('suffix')
        groups.setdefault(base, []).append((col, suf))

# Defaults (match your example: exmin/exmax -> None; min/max -> NaN)
defaults = {'exmin': None, 'min': np.nan, 'max': np.nan, 'exmax': None}

# Build the new dict-valued columns and drop originals
cols_to_drop = []
for base, pairs in groups.items():
    # Start with defaults for all rows
    val_df = pd.DataFrame({k: defaults[k] for k in defaults}, index=df.index)

    # Fill with actual values when present
    for col, suf in pairs:
        key = suffix_map[suf]
        val_df[key] = df[col]
        cols_to_drop.append(col)

    # Create the dict per row in the requested order
    df[base] = val_df.apply(lambda r: {
        'exmin': r['exmin'],
        'min': r['min'],
        'max': r['max'],
        'exmax': r['exmax']
    }, axis=1)
df.drop(columns=cols_to_drop, inplace=True)
# Mapping dictionary: man → auto
# ✅ Map auto → man
map_auto_to_man = {
    'corolla_lip_length': 'corolla_lip',
    'female_calyx_length_lobes': 'female_calyx_lobes_length',
    'calyx_lobe_width_outer': 'outer_calyx_lobes_width',
    'female_calyx_width_lobes': 'female_calyx_lobes_width',
    'calyx_lobe_length_inner': 'inner_calyx_lobes_length',
    'calyx_lobe_width_inner': 'inner_calyx_lobes_width',
    'lower_corolla_length': 'lower_corolla',
    'male_calyx_length_lobes': 'male_calyx_lobes_length',
    'male_calyx_width_lobes': 'male_calyx_lobes_width',
    'male_corolla_lobes_length': 'male_corrola_lobes_length',
    'male_corolla_lobes_width': 'male_corrola_lobes_width',
    'calyx_lobe_length_outer': 'outer_calyx_lobes_length',
    'perianth_length_inner': 'perianth_inner_length',
    'perianth_width_inner': 'perianth_inner_width',
    'perianth_length_outer': 'perianth_outer_length',
    'perianth_width_outer': 'perianth_outer_width',
    'pistallate_corolla_length': 'pistillate_corolla_length',
    'pistallate_corolla_tube_length': 'pistillate_corolla_tube_length',
    'pistallate_corolla_tube_width': 'pistillate_corolla_tube_width',
    'pistallate_pedicel_length': 'pistillate_pedicel_length',
    'pistallate_pedicel_width': 'pistillate_pedicel_width',
    'pistallate_penduncle_length': 'pistillate_peduncle_length',
    'pistallate_penduncle_width': 'pistillate_peduncle_width',
    'pistallate_perianth_tube_length': 'pistillate_perianth_tube_length',
    'upper_corolla_length': 'upper_corolla'
}

# Apply the renaming
df = df.rename(columns=map_auto_to_man)

# ✅ Verification
print("✅ Columns renamed from auto → man convention.")
print("Renamed columns now in df:")
print([v for v in map_auto_to_man.values() if v in df.columns])

# Remove the original component columns



✅ Columns renamed from auto → man convention.
Renamed columns now in df:
['corolla_lip', 'female_calyx_lobes_length', 'outer_calyx_lobes_width', 'female_calyx_lobes_width', 'inner_calyx_lobes_length', 'inner_calyx_lobes_width', 'lower_corolla', 'male_calyx_lobes_length', 'male_calyx_lobes_width', 'male_corrola_lobes_length', 'male_corrola_lobes_width', 'outer_calyx_lobes_length', 'perianth_inner_length', 'perianth_inner_width', 'perianth_outer_length', 'perianth_outer_width', 'pistillate_corolla_length', 'pistillate_corolla_tube_length', 'pistillate_corolla_tube_width', 'pistillate_pedicel_length', 'pistillate_pedicel_width', 'pistillate_peduncle_length', 'pistillate_peduncle_width', 'pistillate_perianth_tube_length', 'upper_corolla']


In [6]:
df

,family,genus,species,infraspecific_epithet,common_name,hawaiian_name,wagner_pg_number,description,life_form_type,stem_hair_type,...,calyx_lobes_length,pistallate_perianth_lobes_length,staminate_pedicel_length,pistallate_pedicel_length,male_calyx_length_lobes,female_calyx_length_lobes,seeds_perfruit,seed_width,juvenile_leaf_length,juvenile_leaf_width
0,Aizoaceae,Trianthema,portulacastrum,None,None,None,None,Dicots,[PERENNIAL_HERB],"[GLABROUS, PUBERULENT]",...,"{'exmin': None, 'min': 4.0, 'max': 5.0, 'exmax...","{'exmin': None, 'min': 4.0, 'max': 5.0, 'exmax...","{'exmin': None, 'min': 0.0, 'max': 0.0, 'exmax...","{'exmin': None, 'min': 0.0, 'max': 0.0, 'exmax...","{'exmin': None, 'min': 4.0, 'max': 5.0, 'exmax...","{'exmin': None, 'min': 4.0, 'max': 5.0, 'exmax...","{'exmin': None, 'min': 1.0, 'max': None, 'exma...","{'exmin': None, 'min': 0.15, 'max': 0.2, 'exma...","{'exmin': None, 'min': 10.0, 'max': 20.0, 'exm...","{'exmin': None, 'min': 4.0, 'max': 20.0, 'exma..."


In [15]:
for i in man_df_test.columns:
    try:
        print(i, df[i].values[0], man_df_test[i].values[0])
    except:
        print('Automation Missing: ', i)

family Aizoaceae Aizoaceae
genus Trianthema Trianthema
species portulacastrum portulacastrum
common_name None no common name
wagner_pg_number None pg 178-179
description Dicots ['DICOTS']
infraspecific_epithet None nan
stem_hair_type ['GLABROUS', 'PUBERULENT'] ['GLABROUS (SMOOTH)', 'PUBERULENT']
phyllotaxy_type ['OPPOSITE'] ['OPPOSITE']
Automation Missing:  leaf_hair_description
Automation Missing:  leaf_hair_upper_description
Automation Missing:  leaf_hair_lower_description
breeding_type None ['MONOECIOUS']
inflorescence_type ['AXILLARY', 'SOLITARY'] ['SOLITARY', 'CLUSTERS']
ray_color None nan
floret_color None nan
spathe_color None nan
perianth_outer_color None nan
perianth_inner_color pink or white nan
perianth_color None nan
labellum_color None nan
corolla_type None ['ABSENT']
staminate_corolla_type [None None] nan
pistillate_corolla_type [None None] nan
corolla_color None nan
fruit_type ['CAPSULE'] ['NAN']
ploidy ['2n'] 2n
chromosome_number [26, 28] 26, 28
average_chromosome_numbe

In [8]:
common_columns = df.columns.intersection(man_df_test.columns)
common_columns

Index(['family', 'genus', 'species', 'infraspecific_epithet', 'common_name',
       'hawaiian_name', 'wagner_pg_number', 'description', 'life_form_type',
       'stem_hair_type',
       ...
       'leaf_length', 'leaf_width', 'petioles', 'pedicel_length',
       'tepal_length', 'calyx_lobes_length', 'seeds_perfruit', 'seed_width',
       'juvenile_leaf_length', 'juvenile_leaf_width'],
      dtype='object', length=147)

In [16]:
# Columns not shared between df and man_df_test
diff_df = df.columns.difference(man_df_test.columns)
diff_man = man_df_test.columns.difference(df.columns)

print("Columns unique to df:")
print((list(diff_df)))

print("\nColumns unique to man_df_test:")
print(list(diff_man))


Columns unique to df:
['female_calyx_tube_length', 'lower_corolla_lip_length', 'male_calyx_lobes_width_inner', 'pistallate_bract_length', 'pistallate_perianth_lobes_length', 'staminate_bract_length', 'staminate_pedicel_length', 'upper_corolla_lip_length']

Columns unique to man_df_test:
['leaf_hair_description', 'leaf_hair_lower_description', 'leaf_hair_upper_description', 'species_id']


In [17]:
for i in man_df_test.columns:
    try:
        print(i, df[i].values[0], man_df_test[i].values[0])
    except:
        print('Automation Missing: ', i)

family Aizoaceae Aizoaceae
genus Trianthema Trianthema
species portulacastrum portulacastrum
common_name None no common name
wagner_pg_number None pg 178-179
description Dicots ['DICOTS']
infraspecific_epithet None nan
stem_hair_type ['GLABROUS', 'PUBERULENT'] ['GLABROUS (SMOOTH)', 'PUBERULENT']
phyllotaxy_type ['OPPOSITE'] ['OPPOSITE']
Automation Missing:  leaf_hair_description
Automation Missing:  leaf_hair_upper_description
Automation Missing:  leaf_hair_lower_description
breeding_type None ['MONOECIOUS']
inflorescence_type ['AXILLARY', 'SOLITARY'] ['SOLITARY', 'CLUSTERS']
ray_color None nan
floret_color None nan
spathe_color None nan
perianth_outer_color None nan
perianth_inner_color pink or white nan
perianth_color None nan
labellum_color None nan
corolla_type None ['ABSENT']
staminate_corolla_type [None None] nan
pistillate_corolla_type [None None] nan
corolla_color None nan
fruit_type ['CAPSULE'] ['NAN']
ploidy ['2n'] 2n
chromosome_number [26, 28] 26, 28
average_chromosome_numbe

In [23]:
import pandas as pd

# ✅ Find shared columns between the two dataframes
shared_cols = df.columns.intersection(man_df_test.columns)

print(f"Number of shared columns: {len(shared_cols)}")
print("Example shared columns:", list(shared_cols)[:10])  # preview a few

# print("✅ Merge complete. Final shape:", merged_df.shape)


Number of shared columns: 172
Example shared columns: ['family', 'genus', 'species', 'infraspecific_epithet', 'common_name', 'hawaiian_name', 'wagner_pg_number', 'description', 'life_form_type', 'stem_hair_type']


In [25]:
import pandas as pd

# --- 1️⃣ Ensure all column names are strings ---
df.columns = df.columns.astype(str)
man_df_test.columns = man_df_test.columns.astype(str)

# --- 2️⃣ Remove duplicate column names ---
df = df.loc[:, ~df.columns.duplicated()]
man_df_test = man_df_test.loc[:, ~man_df_test.columns.duplicated()]

# --- 3️⃣ Concatenate on shared columns only ---
final = pd.concat(
    [df, man_df_test],
    join="inner",       # keep only common columns
    ignore_index=True   # reset index
)

print("✅ Successfully concatenated DataFrames.")
print("Shape:", final.shape)
print("Columns:", list(final.columns))


✅ Successfully concatenated DataFrames.
Shape: (2, 172)
Columns: ['family', 'genus', 'species', 'infraspecific_epithet', 'common_name', 'hawaiian_name', 'wagner_pg_number', 'description', 'life_form_type', 'stem_hair_type', 'leaf_type', 'leaf_shape_type', 'leaf_margin_type', 'phyllotaxy_type', 'leaflets_leaf_type', 'leaflets_leaf_shape_type', 'leaflets_leaf_margin_type', 'leaf_hair_type', 'leaf_hair_upper_type', 'leaf_hair_lower_type', 'leaflets_leaf_length', 'leaflets_leaf_width', 'inflorescence_type', 'peduncle_length', 'peduncle_width', 'pedicel_width', 'spathe_length', 'spathe_width', 'spadix_length', 'umbellet_length', 'hypanthium_length', 'hypanthium_width', 'rachis_length', 'rachis_diameter', 'inflorescence_flower_length', 'inflorescence_flower_width', 'staminate_inflorescence_length', 'staminate_inflorescence_width', 'pistillate_inflorescence_length', 'pistillate_inflorescence_width', 'breeding_type', 'corolla_type', 'corolla_color', 'perianth_color', 'perianth_outer_color', 'p

In [28]:
final.to_csv('/Users/williamharrigan/Desktop/wagner_auto_vs_manual_comparison.csv', index=False)
